# Setup

In [1]:
import io
import pandas as pd
import numpy as np 

import tqdm
import warnings

from matplotlib import pyplot as plt

In [2]:
import boto3

bucket_name = "dev-datamermaid-sm-sources"
prefix = "coralnet-public-images"
s3 = boto3.client("s3")

In [3]:
#Map to CoralNet labelspace
import requests 

mapping_endpoint ="https://api.datamermaid.org/v1/classification/labelmappings/?provider=CoralNet" 

response = requests.get(mapping_endpoint)
data = response.json()
labelset = data["results"]

while data["next"]:
    response = requests.get(data["next"])
    data = response.json()
    labelset.extend(data["results"])
label_mapping = {
    int(label["provider_id"]): label["benthic_attribute_name"] for label in labelset
}

In [4]:
from mermaidseg.datasets.concepts import initialize_benthic_hierarchy
hierarchy_dict = initialize_benthic_hierarchy()

# Data

In [5]:
df_labelset100 = pd.read_csv("dataframes/mapped_to_mermaid_attributes.csv")
df_labelset100_top = df_labelset100[df_labelset100["top100"]=="1"]

# Identify valid sources as one that fit the below requirements and have been labeled as ToKeep by Tiela 

In [6]:
s3_key = f"{prefix}/temporary/df_source_EDA.csv"
obj = s3.get_object(Bucket=bucket_name, Key=s3_key)
df_source = pd.read_csv(obj["Body"])

In [7]:
df_source.head()

,source_id,image_count,annotations_count,num_labels,num_labels_MERMAID,num_labels_MERMAID_top100
0,23,750,15000,20,16,14
1,57,82,8200,27,20,16
2,69,100,4000,5,5,5
3,70,300,12000,16,16,13
4,109,3942,197100,78,51,30


In [8]:
num_images_threshold = 100
num_annotations_threshold = 1000
num_labels_MERMAID_threshold = 10

source_mask = ((df_source['image_count'] > num_images_threshold) & 
                                        (df_source['annotations_count'] >= num_annotations_threshold) & 
                                        (df_source['num_labels_MERMAID_top100'] >= num_labels_MERMAID_threshold))
                                        
print(f"Sources with all updated (example) requirements (MERMAID): {source_mask.sum()}")

df_source.loc[source_mask,["image_count", "annotations_count"]].sum()

Sources with all updated (example) requirements (MERMAID): 284


image_count            611056
annotations_count    15527191
dtype: int64

In [9]:
df_source_quality = pd.read_csv("dataframes/coralnet_source_quality.csv") # The file can be downloaded on the above link as well

In [10]:
source_mask = ((df_source['image_count'] > num_images_threshold) & 
                                        (df_source['annotations_count'] >= num_annotations_threshold) & 
                                        (df_source['num_labels_MERMAID_top100'] >= num_labels_MERMAID_threshold))
                                        
source_quality_list = df_source_quality[df_source_quality["ToKeep"]=="Yes"]["Source ID"].tolist()
df_source_subset = df_source[source_mask]
df_source_subset = df_source_subset[df_source_subset["source_id"].isin(source_quality_list)]

In [11]:
df_source_subset

,source_id,image_count,annotations_count,num_labels,num_labels_MERMAID,num_labels_MERMAID_top100
73,1645,3000,60000,72,58,54
84,1776,448,13440,33,22,15
86,1783,1638,49140,40,29,21
98,1968,184,41400,138,27,13
113,2055,300,3000,38,32,32
...,...,...,...,...,...,...
1134,6931,1468,73400,51,47,46
1135,6932,2933,146650,52,48,47
1153,6968,406,10282,50,34,29
1176,7054,291,14550,35,22,15


In [12]:
df_source_subset[["image_count", "annotations_count"]].sum()

image_count           157121
annotations_count    4593275
dtype: int64

# Get Label Distributions

In [13]:
s3_key = f"{prefix}/temporary/df_label_source_MERMAID_Top100_EDA.csv"
obj = s3.get_object(Bucket=bucket_name, Key=s3_key)
df_source_labels = pd.read_csv(obj["Body"])

In [14]:
df_source_labels = df_source_labels[df_source_labels["MERMAID Label Top100"]!="No Top Level Category"]

In [15]:
df_source_labels.shape

(13663, 4)

In [16]:
df_source_labels.head()

,MERMAID Label Top100,source_id,num_annotations,num_images
0,Acanthaster planci,109,12,3
1,Acanthaster planci,155,2,2
2,Acanthaster planci,554,6,4
3,Acanthaster planci,2112,2,2
4,Acanthaster planci,2236,19,16


In [17]:
df_source_labels_eligible = df_source_labels[df_source_labels["source_id"].isin(df_source_subset["source_id"])]

In [18]:
df_source_labels_eligible

,MERMAID Label Top100,source_id,num_annotations,num_images
11,Acanthaster planci,4710,1,1
12,Acanthaster planci,4721,1,1
18,Acanthastrea,1645,20,15
19,Acanthastrea,2297,19,18
22,Acanthastrea,2978,1,1
...,...,...,...,...
14654,Zoanthid,6920,1,1
14655,Zoanthid,6921,5,4
14656,Zoanthid,6922,7,7
14657,Zoanthid,6923,17,13


In [19]:
df_labels = df_source_labels_eligible.groupby(["MERMAID Label Top100"], as_index=False).agg({"num_annotations": "sum", "num_images": "sum", "source_id": "nunique"}).rename(columns={"source_id": "num_sources"})
df_labels.head()

,MERMAID Label Top100,num_annotations,num_images,num_sources
0,Acanthaster planci,2,2,2
1,Acanthastrea,1437,1021,41
2,Acropora,245590,37949,54
3,Acropora cervicornis,291,71,6
4,Acropora palmata,179,17,3


In [20]:
num_images_threshold = 100
num_annotations_threshold = 1000
num_sources_threshold = 3
df_labels_subset = df_labels[(df_labels["num_annotations"] >= num_annotations_threshold)*
                             (df_labels["num_images"] >= num_images_threshold)*
                             (df_labels["num_sources"] >= num_sources_threshold)]
df_labels_subset

,MERMAID Label Top100,num_annotations,num_images,num_sources
1,Acanthastrea,1437,1021,41
2,Acropora,245590,37949,54
6,Agaricia agaricites,8534,3041,7
10,Alveopora,1178,117,18
12,Astrea,1453,599,17
...,...,...,...,...
101,Symphyllia,1538,919,40
105,Turbinaria-algae,1566,567,16
106,Turbinaria-coral,6124,1820,44
107,Turf algae,774789,84240,36


In [21]:
df_labels_subset.describe()

,num_annotations,num_images,num_sources
count,71.000000,71.000000,71.000000
mean,46821.295775,9591.352113,34.239437
std,114493.453450,15741.973584,17.165967
min,1026.000000,117.000000,4.000000
25%,3179.500000,1096.000000,18.500000
50%,6050.000000,3237.000000,39.000000
75%,31551.000000,9152.000000,48.000000
max,774789.000000,84240.000000,65.000000


We are continuing with only these 72 labels for now

In [22]:
df_source_labels_eligible = df_source_labels_eligible[df_source_labels_eligible["MERMAID Label Top100"].isin(df_labels_subset["MERMAID Label Top100"])]
df_source_labels_eligible

,MERMAID Label Top100,source_id,num_annotations,num_images
18,Acanthastrea,1645,20,15
19,Acanthastrea,2297,19,18
22,Acanthastrea,2978,1,1
24,Acanthastrea,3058,9,9
28,Acanthastrea,3401,24,21
...,...,...,...,...
14654,Zoanthid,6920,1,1
14655,Zoanthid,6921,5,4
14656,Zoanthid,6922,7,7
14657,Zoanthid,6923,17,13


In [23]:
df_source_labels_eligible[["MERMAID Label Top100", "source_id"]].nunique()

MERMAID Label Top100    71
source_id               67
dtype: int64

In [24]:
print("We are finally left with", df_source_labels_eligible[["num_annotations"]].sum().item(), "annotations and", 
df_source_labels_eligible[["MERMAID Label Top100"]].nunique().item(), "labels and", df_source_labels_eligible[["source_id"]].nunique().item(), "sources.")

We are finally left with 3324312 annotations and 71 labels and 67 sources.


In [25]:
df_source_labels_eligible.groupby('source_id')[['num_annotations']].sum().describe()

,num_annotations
count,67.000000
mean,49616.597015
std,69648.549687
min,2730.000000
25%,9801.000000
50%,23802.000000
75%,60294.000000
max,392180.000000


# Split Optimization

In [60]:
# import numpy as np
# import pandas as pd
# from collections import defaultdict

# def constrained_stratified_group_split_from_aggregated(
#     df_agg,
#     source_col='Source ID',
#     label_col='Label ID',
#     count_col='Num Annotations',  # or 'Num Images' depending on what you want to balance
#     train_ratio=0.7,
#     val_ratio=0.15,
#     test_ratio=0.15,
#     n_iterations=5000,
#     random_state=42,
#     verbose=True
# ):
#     """
#     Split pre-aggregated dataset with:
#     - No source leakage across splits
#     - Stratified labels based on annotation counts
#     - Respects source-level grouping
    
#     Parameters:
#     -----------
#     df_agg : pd.DataFrame
#         Pre-aggregated dataframe with columns: source, label, count
#     source_col : str
#         Column identifying unique sources (e.g., 'Source ID')
#     label_col : str
#         Column with target labels (e.g., 'Label ID')
#     count_col : str
#         Column with counts to balance (e.g., 'Num Annotations')
#     train_ratio, val_ratio, test_ratio : float
#         Desired split proportions (should sum to 1)
#     n_iterations : int
#         Number of random assignments to try (more = better balance)
#     random_state : int
#         Random seed for reproducibility
#     verbose : bool
#         Print progress and statistics
#     """
#     np.random.seed(random_state)
    
#     # Validate inputs
#     assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-10, "Ratios must sum to 1"
    
#     # 1. Get unique sources and labels
#     sources = df_agg[source_col].unique()
#     unique_labels = df_agg[label_col].unique()
#     n_sources = len(sources)
#     n_labels = len(unique_labels)
    
#     if verbose:
#         print(f"Found {n_sources} unique sources")
#         print(f"Found {n_labels} unique labels")
#         print(f"Total {count_col}: {df_agg[count_col].sum():,}")
    
#     # 2. Create source-label count matrix
#     # Initialize matrix: sources × labels
#     label_matrix = np.zeros((n_sources, n_labels))
#     source_to_idx = {source: i for i, source in enumerate(sources)}
#     label_to_idx = {label: j for j, label in enumerate(unique_labels)}
    
#     for _, row in df_agg.iterrows():
#         i = source_to_idx[row[source_col]]
#         j = label_to_idx[row[label_col]]
#         label_matrix[i, j] = row[count_col]
    
#     # 3. Calculate source sizes (total annotations per source)
#     source_sizes = label_matrix.sum(axis=1)
    
#     # 4. Calculate global label distribution (target proportions)
#     total_annotations = label_matrix.sum()
#     global_label_dist = label_matrix.sum(axis=0) / total_annotations
    
#     if verbose:
#         print(f"\nGlobal label distribution (top 5):")
#         for j in np.argsort(-global_label_dist)[:5]:
#             print(f"  Label {unique_labels[j]}: {global_label_dist[j]:.2%}")
    
#     # 5. Iterative random assignment with scoring
#     best_score = np.inf
#     best_assignment = None
#     best_split_counts = None
    
#     target_sizes = np.array([train_ratio, val_ratio, test_ratio]) * total_annotations
    
#     if verbose:
#         print(f"\nTarget split sizes:")
#         print(f"  Train: {target_sizes[0]:,.0f} annotations ({train_ratio:.1%})")
#         print(f"  Val:   {target_sizes[1]:,.0f} annotations ({val_ratio:.1%})")
#         print(f"  Test:  {target_sizes[2]:,.0f} annotations ({test_ratio:.1%})")
#         print(f"\nRunning {n_iterations} iterations...")
    
#     for iteration in range(n_iterations):
#         if verbose and iteration % 1000 == 0:
#             print(f"  Iteration {iteration}/{n_iterations}")
        
#         # Randomly shuffle sources for greedy assignment
#         shuffled_idx = np.random.permutation(n_sources)
#         assignments = np.zeros(n_sources, dtype=int)
#         split_counts = np.zeros((3, n_labels))
#         split_sizes = np.zeros(3)
        
#         # Greedy assignment
#         for idx in shuffled_idx:
#             source_label_counts = label_matrix[idx]
#             source_size = source_sizes[idx]
            
#             # Calculate imbalance if added to each split
#             scores = []
#             for split_id in range(3):
#                 current_size = split_sizes[split_id]
                
#                 # Penalize if split would exceed target by >20%
#                 if current_size + source_size > target_sizes[split_id] * 1.2:
#                     scores.append(np.inf)
#                 else:
#                     new_counts = split_counts[split_id] + source_label_counts
#                     new_size = current_size + source_size
                    
#                     # Expected counts for this split size
#                     expected = new_size * global_label_dist
                    
#                     # Chi-square distance from expected proportions
#                     # Add small epsilon to avoid division by zero
#                     score = np.sum((new_counts - expected)**2 / (expected + 1e-10))
                    
#                     # Add penalty for deviating from target size
#                     size_penalty = abs(new_size - target_sizes[split_id]) / target_sizes[split_id] * 10
#                     score += size_penalty
                    
#                     scores.append(score)
            
#             # Assign to split with minimum score
#             best_split = np.argmin(scores)
#             assignments[idx] = best_split
#             split_counts[best_split] += source_label_counts
#             split_sizes[best_split] += source_size
        
#         # Calculate overall score (lower is better)
#         overall_score = 0
#         for split_id in range(3):
#             expected = split_sizes[split_id] * global_label_dist
#             overall_score += np.sum((split_counts[split_id] - expected)**2 / (expected + 1e-10))

#         if overall_score < best_score:
#             best_score = overall_score
#             best_assignment = assignments.copy()
#             best_split_counts = split_counts.copy()
            
#             if verbose:
#                 print(f"    New best score: {best_score:.2f}")
    
#     if verbose:
#         print(f"\nOptimization complete after {n_iterations} iterations! Best score: {best_score:.2f}")
    
#     # 6. Create split mapping
#     split_map = {0: 'train', 1: 'val', 2: 'test'}
#     source_to_split = {source: split_map[best_assignment[i]] 
#                        for i, source in enumerate(sources)}
    
#     # 7. Add split column to dataframe
#     df_result = df_agg.copy()
#     df_result['split'] = df_result[source_col].map(source_to_split)
    
#     # 8. Print summary statistics
#     if verbose:
#         print("\n" + "="*50)
#         print("SPLIT SUMMARY")
#         print("="*50)
        
#         for split_name in ['train', 'val', 'test']:
#             split_df = df_result[df_result['split'] == split_name]
#             split_annotations = split_df[count_col].sum()
#             split_sources = split_df[source_col].nunique()
            
#             print(f"\n{split_name.upper()}:")
#             print(f"  Annotations: {split_annotations:,} ({split_annotations/total_annotations:.2%})")
#             print(f"  Sources: {split_sources} ({split_sources/n_sources:.2%})")
            
#             # Label distribution check
#             label_dist = split_df.groupby(label_col)[count_col].sum()
#             label_dist_pct = label_dist / label_dist.sum()
#             print(f"  Top 3 labels: {', '.join([f'{lbl}: {pct:.2%}' for lbl, pct in label_dist_pct.nlargest(3).items()])}")
    
#     return df_result

In [61]:
import numpy as np
import pandas as pd
from collections import defaultdict

def constrained_stratified_group_split_from_aggregated(
    df_agg,
    source_col='Source ID',
    label_col='Label ID',
    count_col='Num Annotations',  # or 'Num Images' depending on what you want to balance
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    n_iterations=5000,
    random_state=42,
    verbose=True
):
    """
    Split pre-aggregated dataset with:
    - No source leakage across splits
    - Stratified labels based on annotation counts
    - Respects source-level grouping
    
    Parameters:
    -----------
    df_agg : pd.DataFrame
        Pre-aggregated dataframe with columns: source, label, count
    source_col : str
        Column identifying unique sources (e.g., 'Source ID')
    label_col : str
        Column with target labels (e.g., 'Label ID')
    count_col : str
        Column with counts to balance (e.g., 'Num Annotations')
    train_ratio, val_ratio, test_ratio : float
        Desired split proportions (should sum to 1)
    n_iterations : int
        Number of random assignments to try (more = better balance)
    random_state : int
        Random seed for reproducibility
    verbose : bool
        Print progress and statistics
    """
    np.random.seed(random_state)
    
    # Validate inputs
    assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-10, "Ratios must sum to 1"
    
    # 1. Get unique sources and labels
    sources = df_agg[source_col].unique()
    unique_labels = df_agg[label_col].unique()
    n_sources = len(sources)
    n_labels = len(unique_labels)
    
    if verbose:
        print(f"Found {n_sources} unique sources")
        print(f"Found {n_labels} unique labels")
        print(f"Total {count_col}: {df_agg[count_col].sum():,}")
    
    # 2. Create source-label count matrix
    # Initialize matrix: sources × labels
    label_matrix = np.zeros((n_sources, n_labels))
    source_to_idx = {source: i for i, source in enumerate(sources)}
    # Reorder labels by rarity (rare labels first) to help the optimizer
    label_source_counts = (
        df_agg.groupby(label_col)[source_col]
        .nunique()
        .reindex(unique_labels, fill_value=0)
    )
    unique_labels = label_source_counts.sort_values().index.to_numpy()
    label_to_idx = {label: j for j, label in enumerate(unique_labels)}

    # Hard feasibility check:
    # with group-based splitting, each label must exist in >= 3 different sources
    # to be present in train/val/test simultaneously.
    impossible_labels = label_source_counts[label_source_counts < 3]
    if not impossible_labels.empty:
        preview = ", ".join(
            f"{lbl} ({cnt} sources)" for lbl, cnt in impossible_labels.head(10).items()
        )
        raise ValueError(
            "Cannot guarantee every label in every split. "
            f"These labels appear in fewer than 3 sources: {preview}"
            + (" ..." if len(impossible_labels) > 10 else "")
        )
    
    for _, row in df_agg.iterrows():
        i = source_to_idx[row[source_col]]
        j = label_to_idx[row[label_col]]
        label_matrix[i, j] = row[count_col]
    
    # 3. Calculate source sizes (total annotations per source)
    source_sizes = label_matrix.sum(axis=1)
    
    # 4. Calculate global label distribution (target proportions)
    total_annotations = label_matrix.sum()
    global_label_dist = label_matrix.sum(axis=0) / total_annotations
    
    if verbose:
        print(f"\nGlobal label distribution (top 5):")
        for j in np.argsort(-global_label_dist)[:5]:
            print(f"  Label {unique_labels[j]}: {global_label_dist[j]:.2%}")
    
    # 5. Iterative random assignment with scoring
    best_score = np.inf
    best_assignment = None
    best_split_counts = None
    
    target_sizes = np.array([train_ratio, val_ratio, test_ratio]) * total_annotations
    
    if verbose:
        print(f"\nTarget split sizes:")
        print(f"  Train: {target_sizes[0]:,.0f} annotations ({train_ratio:.1%})")
        print(f"  Val:   {target_sizes[1]:,.0f} annotations ({val_ratio:.1%})")
        print(f"  Test:  {target_sizes[2]:,.0f} annotations ({test_ratio:.1%})")
        print(f"\nRunning {n_iterations} iterations...")
    
    for iteration in range(n_iterations):
        if verbose and iteration % 1000 == 0:
            print(f"  Iteration {iteration}/{n_iterations}")
        
        # Randomly shuffle sources for greedy assignment
        shuffled_idx = np.random.permutation(n_sources)
        assignments = np.zeros(n_sources, dtype=int)
        split_counts = np.zeros((3, n_labels))
        split_sizes = np.zeros(3)
        
        # Greedy assignment
        for idx in shuffled_idx:
            source_label_counts = label_matrix[idx]
            source_size = source_sizes[idx]
            
            # Calculate imbalance if added to each split
            scores = []
            for split_id in range(3):
                current_size = split_sizes[split_id]
                
                # Penalize if split would exceed target by >20%
                if current_size + source_size > target_sizes[split_id] * 1.2:
                    scores.append(np.inf)
                else:
                    new_counts = split_counts[split_id] + source_label_counts
                    new_size = current_size + source_size
                    
                    # Expected counts for this split size
                    expected = new_size * global_label_dist
                    
                    # Chi-square distance from expected proportions
                    # Add small epsilon to avoid division by zero
                    score = np.sum((new_counts - expected)**2 / (expected + 1e-10))
                    
                    # Add penalty for deviating from target size
                    size_penalty = abs(new_size - target_sizes[split_id]) / target_sizes[split_id] * 10
                    score += size_penalty
                    
                    scores.append(score)
            
            # Assign to split with minimum score
            best_split = np.argmin(scores)
            assignments[idx] = best_split
            split_counts[best_split] += source_label_counts
            split_sizes[best_split] += source_size
        
        # Calculate overall score (lower is better)
        overall_score = 0
        for split_id in range(3):
            expected = split_sizes[split_id] * global_label_dist
            overall_score += np.sum((split_counts[split_id] - expected)**2 / (expected + 1e-10))

        if overall_score < best_score:
            best_score = overall_score
            best_assignment = assignments.copy()
            best_split_counts = split_counts.copy()
            
            if verbose:
                print(f"    New best score: {best_score:.2f}")
    
    if verbose:
        print(f"\nOptimization complete after {n_iterations} iterations! Best score: {best_score:.2f}")
    
    # 6. Create split mapping
    split_map = {0: 'train', 1: 'val', 2: 'test'}
    source_to_split = {source: split_map[best_assignment[i]] 
                       for i, source in enumerate(sources)}
    
    # 7. Add split column to dataframe
    df_result = df_agg.copy()
    df_result['split'] = df_result[source_col].map(source_to_split)
    
    # 8. Print summary statistics
    if verbose:
        print("\n" + "="*50)
        print("SPLIT SUMMARY")
        print("="*50)
        
        for split_name in ['train', 'val', 'test']:
            split_df = df_result[df_result['split'] == split_name]
            split_annotations = split_df[count_col].sum()
            split_sources = split_df[source_col].nunique()
            
            print(f"\n{split_name.upper()}:")
            print(f"  Annotations: {split_annotations:,} ({split_annotations/total_annotations:.2%})")
            print(f"  Sources: {split_sources} ({split_sources/n_sources:.2%})")
            
            # Label distribution check
            label_dist = split_df.groupby(label_col)[count_col].sum()
            label_dist_pct = label_dist / label_dist.sum()
            print(f"  Top 3 labels: {', '.join([f'{lbl}: {pct:.2%}' for lbl, pct in label_dist_pct.nlargest(3).items()])}")
    
    return df_result

- Random seed 43 with 10k iterations and a 0.7, 0.15, 0.15 split with the old function gives a solid result
- Random seed 43 with 15k iterations and a 0.64, 0.18, 0.18 split with the second (updated function) gives a result that includes all labels in all splits

In [73]:
df_with_splits = constrained_stratified_group_split_from_aggregated(
    df_agg=df_source_labels_eligible,  
    source_col='source_id',
    label_col='MERMAID Label Top100',
    count_col='num_annotations',  
    train_ratio=0.64,
    val_ratio=0.18,
    test_ratio=0.18,
    n_iterations=15000,
    random_state=43
)

Found 67 unique sources
Found 71 unique labels
Total num_annotations: 3,324,312

Global label distribution (top 5):
  Label Turf algae: 23.31%
  Label Crustose coralline algae: 12.91%
  Label Sand: 9.04%
  Label Acropora: 7.39%
  Label Porites: 6.66%

Target split sizes:
  Train: 2,127,560 annotations (64.0%)
  Val:   598,376 annotations (18.0%)
  Test:  598,376 annotations (18.0%)

Running 15000 iterations...
  Iteration 0/15000
    New best score: 498648.54
    New best score: 334289.46
    New best score: 333626.14
    New best score: 321498.12
    New best score: 303656.06


    New best score: 243861.45
    New best score: 211332.71
    New best score: 207439.27
    New best score: 197553.79
    New best score: 174734.72
  Iteration 1000/15000
    New best score: 173995.65
    New best score: 171048.76
  Iteration 2000/15000
  Iteration 3000/15000
  Iteration 4000/15000
  Iteration 5000/15000
    New best score: 170415.62
  Iteration 6000/15000
  Iteration 7000/15000
    New best score: 161315.56
  Iteration 8000/15000
  Iteration 9000/15000
  Iteration 10000/15000
  Iteration 11000/15000
  Iteration 12000/15000
  Iteration 13000/15000
  Iteration 14000/15000
    New best score: 126025.83

Optimization complete after 15000 iterations! Best score: 126025.83

SPLIT SUMMARY

TRAIN:
  Annotations: 2,531,431 (76.15%)
  Sources: 34 (50.75%)
  Top 3 labels: Turf algae: 23.51%, Crustose coralline algae: 12.52%, Sand: 9.99%

VAL:
  Annotations: 255,686 (7.69%)
  Sources: 15 (22.39%)
  Top 3 labels: Turf algae: 17.30%, Crustose coralline algae: 10.60%, Porites: 9.3

In [74]:
df_source_splits = df_with_splits.groupby("source_id", as_index=False).agg(
    num_annotations=("num_annotations", "sum"),
    split=("split", "first"),
    num_unique_labels=("MERMAID Label Top100", "nunique"),
)

df_source_splits = df_source_splits.merge(df_source_subset[["source_id", "image_count"]], on="source_id", how="left")

In [69]:
df_source_splits

,source_id,num_annotations,split,num_unique_labels,image_count
0,1645,58095,train,45,3000
1,1776,10017,train,14,448
2,1783,41234,train,15,1638
3,1968,18486,val,8,184
4,2055,2994,train,30,300
...,...,...,...,...,...
62,6931,23802,val,39,1468
63,6932,47923,test,40,2933
64,6968,7915,val,23,406
65,7054,12838,val,14,291


In [79]:
csv_buffer = io.StringIO()
df_source_splits.to_csv(csv_buffer, index=False)

s3_key = f"{prefix}/temporary/df_source_splits.csv"

# Upload to S3
s3.put_object(
    Bucket=bucket_name, Key=s3_key, Body=csv_buffer.getvalue(), ContentType="text/csv"
)

{'ResponseMetadata': {'RequestId': 'N1EWGEA4TCVKDD8F',
  'HostId': 'kvZvLZFBTjF+6NVQIyR56SdWD9kB/y0pM/nfI6KhTMEKKeT06XDGjgSRrfcdpyR/GTbA1eDvwpbX9Qr9Hno/i24UqRwX1tXhGvWeiHxLnmw=',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amz-id-2': 'kvZvLZFBTjF+6NVQIyR56SdWD9kB/y0pM/nfI6KhTMEKKeT06XDGjgSRrfcdpyR/GTbA1eDvwpbX9Qr9Hno/i24UqRwX1tXhGvWeiHxLnmw=',
   'x-amz-request-id': 'N1EWGEA4TCVKDD8F',
   'date': 'Mon, 04 May 2026 12:14:03 GMT',
   'x-amz-server-side-encryption': 'AES256',
   'etag': '"59686e3294465e2c78be72c10b49e93b"',
   'x-amz-checksum-crc32': 'UOJrbQ==',
   'x-amz-checksum-type': 'FULL_OBJECT',
   'content-length': '0',
   'server': 'AmazonS3'},
  'RetryAttempts': 0},
 'ETag': '"59686e3294465e2c78be72c10b49e93b"',
 'ChecksumCRC32': 'UOJrbQ==',
 'ChecksumType': 'FULL_OBJECT',
 'ServerSideEncryption': 'AES256'}

In [76]:
df_source_splits.groupby('split', as_index=False)[['num_annotations', 'image_count']].sum()

,split,num_annotations,image_count
0,test,537195,32548
1,train,2531431,116085
2,val,255686,8488


In [77]:
def validate_aggregated_splits(df_with_splits, source_col='source_id', label_col='MERMAID Label Top100', count_col='num_annotations'):
    """
    Comprehensive validation of the split results
    """
    print("="*60)
    print("VALIDATION RESULTS")
    print("="*60)
    
    # 1. Source disjointness (CRITICAL)
    source_split_counts = df_with_splits.groupby(source_col)['split'].nunique()
    sources_in_multiple_splits = (source_split_counts > 1).sum()
    
    if sources_in_multiple_splits == 0:
        print("✓ PASS: Each source appears in exactly one split")
    else:
        print(f"✗ FAIL: {sources_in_multiple_splits} sources appear in multiple splits!")
        print("  Problematic sources:", source_split_counts[source_split_counts > 1].index.tolist())
    
    # 2. Label distribution similarity
    print("\nLabel Distribution by Split:")
    label_totals = df_with_splits.groupby('split')[count_col].sum()
    global_label_dist = df_with_splits.groupby(label_col)[count_col].sum() / label_totals.sum()

    comparison_df = pd.DataFrame()
    for split in ['train', 'val', 'test']:
        split_data = df_with_splits[df_with_splits['split'] == split]
        split_dist = split_data.groupby(label_col)[count_col].sum() / split_data[count_col].sum()
        comparison_df[split] = split_dist
    
    comparison_df['global'] = global_label_dist
    comparison_df['max_deviation'] = comparison_df[['train', 'val', 'test']].sub(comparison_df['global'], axis=0).abs().max(axis=1)
    
    print(f"  Average label distribution deviation: {comparison_df['max_deviation'].mean():.4f}")
    print(f"  Max label distribution deviation: {comparison_df['max_deviation'].max():.4f}")
    
    # 3. Split sizes
    print("\nSplit Sizes:")
    for split in ['train', 'val', 'test']:
        split_total = df_with_splits[df_with_splits['split'] == split][count_col].sum()
        pct = split_total / label_totals.sum()
        print(f"  {split}: {split_total:,} annotations ({pct:.2%})")
    
    # 4. Coverage check
    print(f"\nTotal unique labels: {df_with_splits[label_col].nunique()}")
    for split in ['train', 'val', 'test']:
        split_labels = df_with_splits[df_with_splits['split'] == split][label_col].nunique()
        coverage = split_labels / df_with_splits[label_col].nunique()
        print(f"  {split} label coverage: {split_labels}/{df_with_splits[label_col].nunique()} ({coverage:.2%})")
    comparison_df = comparison_df[["train", "val", "test"]].div(comparison_df["global"], axis=0)
    comparison_df["std"] = comparison_df[["train", "val", "test"]].std(axis=1)
    return comparison_df

# Run validation
comparison_df = validate_aggregated_splits(df_with_splits)

VALIDATION RESULTS
✓ PASS: Each source appears in exactly one split

Label Distribution by Split:
  Average label distribution deviation: 0.0052
  Max label distribution deviation: 0.0600

Split Sizes:
  train: 2,531,431 annotations (76.15%)
  val: 255,686 annotations (7.69%)
  test: 537,195 annotations (16.16%)

Total unique labels: 71
  train label coverage: 71/71 (100.00%)
  val label coverage: 71/71 (100.00%)
  test label coverage: 71/71 (100.00%)


In [78]:
comparison_df.sort_values("std", ascending=False)

,train,val,test,std
MERMAID Label Top100,,,,
Alveopora,0.028984,12.659395,0.026266,7.292956
Euphyllia,0.001739,0.086103,6.139100,3.519306
Sponge,0.762314,3.072670,1.133531,1.240686
Pachyseris,0.820495,2.685101,1.043836,1.018200
Mycedium,0.843276,2.548550,1.001478,0.942197
...,...,...,...,...
Symphyllia,1.016076,1.005971,0.921402,0.051989
Goniastrea,0.979753,1.065539,1.064214,0.049150
Favia,1.013195,0.932027,0.970173,0.040609
